# Fundamentals of biochemical modelling in Julia

In this workshop, we will build a model of gene expression step by step, starting with the production and degradation of mRNA and ending with a small autoregulatory feedback loop. We will use our mechanistic model to predict both how expression changes over time and how it varies between cells. We will then explore how these predictions change when we add promoter switching, RNA processing, regulation and cell cycle.

### What is Julia?

<img src="figures/Julia_ecosystem.svg" alt="Julia Ecosystem" width="900"/>

<sub>Figure adapted from [Roesch, E., Greener, J.G., MacLean, A.L. et al. Julia for biologists. Nat Methods 20, 655–664 (2023).](https://doi.org/10.1038/s41592-023-01832-z)</sub>

Julia is a relatively young programming language that has become very popular in numerical and scientific computing due to its simplicity and performance. It is  increasingly used to analyse and model biological data, due to its active state-of-the-art package ecosystem (as shown in the figure above). It also boasts a rich infrastructure for scientific computing and differential equations that approaches C/C++ speeds while being significantly more user-friendly. If you have used Python or Matlab before, you should find Julia's syntax very familiar. 

In this notebook we will introduce the package [Catalyst.jl](https://github.com/SciML/Catalyst.jl), which allows us to model chemical reaction networks within Julia. We also make use of [DifferentialEquations.jl](https://docs.sciml.ai/DiffEqDocs/stable/), which provides efficient solvers for most kinds of differential equations one is likely to encounter in scientific modelling.

You can download and install Julia from the [official website](https://julialang.org/). 

### Setting up the environment

Julia's built-in package manager allows us to download and install packages from within Julia. 

**If you already use Julia:** To avoid situations where different projects require different package versions, we recommend running this notebook inside its own environment. The following code will create a new environment in the project directory and install the packages we will need for this project:

In [ ]:
using Pkg             # Load the package manager

Pkg.activate(".")     # Create a new environment in the project folder  

# Install the packages we need - this might take a while
Pkg.add(["Catalyst", 
         "DifferentialEquations", 
         "Plots",
         "Random", 
         "Distributions",
         "Latexify",
         "DiffEqCallbacks"])

Unlike Python, but like R, Julia frequently needs to precompile its packages. This may take a while, so we suggest installing all packages before the workshop starts.

**Note:** Catalyst.jl is currently in active development. To make use of all of its features we recommend running this notebook using Catalyst v16.

## A simple model of gene expression

Let us start with a very simple example where we model the production and degradation of mRNA (which we call $M$) for a single gene. We assume that $M$ is produced at a constant rate $\rho_m$ and degraded with rate $\delta_m$. The network can be visualised schematically as follows:

<img src="figures/model_Poisson.png" alt="Event-driven simulation" width="500"/>

$$G \xrightarrow{\rho_{\mathrm{m}}} G + M,
\qquad M \xrightarrow{\delta_m} \varnothing.$$

| Reaction | Parameter units |
|---|---|
| Transcription | $\rho_{\mathrm{m}}$: min $^{-1}$|
| Degradation | $\delta_M$: min $^{-1}$ |

In the transcription reaction, the gene $G$ appears on both sides because producing an RNA molecule does not consume the gene. The symbol $\varnothing$ represents ``nothing''. Each mRNA molecule has the same degradation rate, so the total rate of degradation increases with the number of molecules present.

We implement this in [Catalyst](https://github.com/SciML/Catalyst.jl) using the `@reaction_network` macro [<sup>*</sup>](#fn1), which takes a list of reactions and converts them into a reaction network in Julia.

<small id="fn1">* A macro manipulates source code to introduce custom syntax into Julia. In this case, the macro `@reaction_network` allows us to specify a reaction network as a list of equations, which then get parsed and converted into ordinary Julia objects. [^](#fn1)</small> 

In [ ]:
using Catalyst # load the package (analogous to Python's `import`)

rn = @reaction_network begin
    ρ_m, G --> G + M    # mRNA production
    δ_m, M --> 0        # mRNA degradation
end

 Catalyst automatically figures out which variables correspond to species and which ones to parameters. <!--If we want to explicitly specify these, we can write -->
<!-- ```julia
@species M(t)
@parameters ρ_m δ_m
```
within the reaction network. The notation `M(t)` tells us that $M$ is a time-dependent variable. -->

Having specified the reactions, we can ask how the amount of mRNA changes over time. One way to describe this is with an ordinary differential equation (ODE). An ODE specifies how quickly a quantity changes at each moment. Here, the rule is simply: rate of change = production − degradation. For one fixed gene
copy, this gives:
$$
\frac{dM}{dt} = \rho_m- \delta_m M
$$
The left-hand side means “how quickly the mRNA amount is changing”. Production adds mRNA at rate $\rho_m$, while degradation removes it at a rate proportional to the amount already present. Given the parameter values and a starting amount, solving this equation produces a curve describing mRNA abundance over time.

We can do this automatically using Catalyst, which can directly convert the symbolic network description into a system of ODEs. This can visualised mathematically with Latexify

In [ ]:
using Latexify
latexify(rn, form = :ode)

If we want to simulate our system, we need to specify model parameters (the transcription rate $\rho_m$ and degradation rate $\delta_m$) and the initial conditions (what number of $G$ and $M$ molecules we start with). 

In [ ]:
# Model parameters are given as a list of parameter-value pairs
# The notation :var indicates that 'var' is the name of a variable.
p = [ :ρ_m => 5, 
      :δ_m => 0.5 ]

# Initial conditions are given as a list of variable-initial value pairs
u0 = [ :G => 1, :M => 0 ]

# Simulation length
tmax = 10.0;

The package DifferentialEquations.jl allows us to numerically solve these ODEs. For this purpose we set up an `ODEProblem` with our equation and the chosen parameters and initial values. DifferentialEquations takes care of choosing the right numerical solvers, tolerances and step sizes, so a simple call to `solve` is all that is necessary to solve our ODE problem.

In [ ]:
using DifferentialEquations

prob_ODE = ODEProblem(rn, u0, (0, tmax), p)
sol_ODE = solve(prob_ODE, saveat=0.1);

We can access the solution variables using bracket notation, as if the solution was a dictionary. Here the colon `:` is used to mark $M$ as the name of a variable, like a string. We used the same notation above when we specified parameters and initial values. 

In [ ]:
sol_ODE[:M]         # Returns an array

The solution times are given by `sol.t`:

In [ ]:
sol_ODE.t           # Another array

We can then easily [plot](https://docs.sciml.ai/DiffEqDocs/stable/basics/plot/) the obtained solution for $M$ using [Plots.jl](https://github.com/JuliaPlots/Plots.jl)

In [ ]:
using Plots

# Set to use svg format for clarity (for performance, switch to png and change the default global DPI for all plots in the notebook)
# default(fmt=:png); default(dpi=120)
default(fmt=:svg)

plot(sol_ODE.t, sol_ODE[:M], 
     xlabel="Time", 
     ylabel="Molecules",
     label="mRNA",
     lw=2)


The curve levels off when production and degradation balance. With our parameter values, this happens
at $M=\rho_m/\delta_m=10$.

### Stochastic simulation

The ODE gives a smooth continuous curve for $M$, which can take values such as $0.3$, $1.8$ or $3.3$ molecules. For this simple model, we can interpret that curve as the average mRNA count across many cells. An average need not be an integer, even though each individual cell contains a whole number of molecules.

This is a good approximation when mRNA numbers are very large, so they stay close to the mean. In many cases, molecule numbers within a cell are rather low, and fluctuate randomly due to molecular noise. These fluctuations can change the behaviour of a cell, leading e.g. to cell differentiation, phenotypic switching or cell death. 

To describe individual cells and their variability, we model individual molecule counts, and view transcription and degradation as discrete events occurring at random times. We achieve this by treating our system as a jump differential equation, where jumps (transcription or degradation) happen at random times, governed by the reaction propensities. A reaction’s propensity describes how likely a reaction is to occur in the next short interval. For degradation, this is $\delta_m M$. The reaction propensity is proportional to the number of mRNA molecules: more molecules provide more opportunities for a degradation event. The discretisation and the reaction propensities are summarised in the table below:

| Reaction | Change in mRNA count | Reaction propensity | Parameter units |
|---|---:|---|---|
| Transcription | $+1$ | $\rho_{\mathrm{m}}$ | $\rho_{\mathrm{m}}$: min $^{-1}$|
| Degradation | $-1$ | $\delta_M M$ | $\delta_M$: min $^{-1}$ |

This can be simulated using the Stochastic Simulation Algorithm (SSA), also known as the [Gillespie algorithm](https://en.wikipedia.org/wiki/Gillespie_algorithm). The Gillespie algorithm samples the waiting time until the next reaction
and which reaction occurs, then advances directly to that event.

<img src="figures/event_driven_figure.png" alt="Event-driven simulation" width="900"/>

We can easily perform stochastic simulations in Julia using [JumpProcesses.jl](https://docs.sciml.ai/JumpProcesses/stable/) (a part of the DifferentialEquations ecosystem). Constructing a `JumpProblem` is as easy as an `ODEProblem`. Again we need the reaction network, initial conditions and parameters.

In [ ]:
jprob = JumpProblem(rn, u0, (0, tmax), p)

We solve this equation as before, using the `solve` function. Again, DifferentialEquations takes care of choosing the correct solver for us. It uses the SSA already by default.

In [ ]:
sol_SSA = solve(jprob);

Plotting the solution we see that mRNA numbers are now discrete and randomly fluctuating, as they should be.

In [ ]:
plt = plot(sol_SSA.t, sol_SSA[:M], 
           xlabel="Time", 
           ylabel="Molecules",
           label="mRNA", 
           linetype=:steppost,
           linewidth=2)

The result of the above simulation is stochastic: if we run `solve` again, we get a different trajectory. For example, we can generate and plot another two stochastic trajectories:

In [ ]:
for i in 1:2
    sol_SSA = solve(jprob)
    plot!(sol_SSA.t, sol_SSA[:M], linetype=:steppost, label="", linewidth=2)
end

plt

We can think of each trajectory as one simulated cell with the same parameters and starting conditions. Repeating the simulation lets us describe the variation between these cells.

**Exercise:**

Try rerunning the cell above multiple times to add extra trajectories to the plot — just by simple visual expection we can gain intuition about the molecule count statistics, such as its mean and variance.

Next, let's compare the stochastic simulation more directly to its deterministic counterpart, i.e., the ODE system we were solving just before. By generating many SSA trajectories and averaging over them we can quantify the time evolution of the mean mRNA counts: 

In [ ]:
using Statistics

# Number of simulations to perform
n_trajectories = 200
# Time points for saving the system state
ts = 0:0.1:tmax
# Matrix to store the count time series from each simulation
ms = Matrix{Int}(undef, n_trajectories, length(ts))

for i in 1:n_trajectories
    sol_SSA = solve(jprob)
    m = sol_SSA(ts, idxs=:M) # record the mRNA count at time t = tmax
    ms[i, :] = m
end

ensemble_mean = vec(mean(ms; dims=1));

We can compare the obtained mean to the ODE system solution:

In [ ]:
p_ensemble = plot(xlabel="Time (min)", ylabel="RNA molecules")
cturq = "#168E92"
cmagenta = "#B23A78"

for i in 1:10
    plot!(p_ensemble, ts, ms[i, :]; seriestype=:steppost,
          color=cturq, alpha=0.2, linewidth=1.5, label=false)
end

plot!(p_ensemble, sol_ODE.t, sol_ODE[:M]; color=cmagenta,
      linestyle=:dash, linewidth=3, label="Deterministic mean")

plot!(p_ensemble, ts, ensemble_mean; color=cturq,
      linewidth=3, label="Mean of $n_trajectories cells")
p_ensemble

We can see that both are in good agreement, albeit individual stochastic trajectories tend to noticeably deviate from the mean.
For this simple linear model, the ODE describes the mean exactly; the small discrepancy here comes from averaging a finite number of simulations. The stochastic simulations additionally describe the variation around that mean.

To obtain an deeper understanding of the stochastic dynamics of a system, we are often interested not only in the summary statistics, but the full probability distribution of molecule counts at certain simulation times. This can be estimated by creating a histogram of mRNA numbers over the SSA trajectories; by taking one count from each trajectory gives a snapshot distribution, analogous to the biological variation we aim to understand in single-cell experiments.

In [ ]:
# Plot the discrete probability distribution
histogram(ms[:, end],         # `end` takes the last saved timepoint corresponding to ts[end]
          bins=0:maximum(ms), # define bins from 0 to the maximum observed mRNA count
          normalize=:probability,
          xlabel="Molecules", 
          ylabel="Probability", 
          label="mRNA",
          color=cturq, alpha=0.8) 

For fun, we can combine the plots above into one:

In [ ]:
# NOTE: the helper function is just for nicer plots -- run this cell but feel free to ignore the internals
using Plots.Measures: mm

function plot_dynamics(counts::AbstractMatrix, _t::Real; 
                      n_trajectories=size(counts, 1), title="", ylabel="mRNA counts", highlight_bar::Bool=true)
    _idx = argmin(abs.(ts .- _t))
    #_idx = 15
    snapshot_values = counts[:, _idx]
    
    p_history = plot(xlabel="Time (min)", ylabel=ylabel,
                     title=title, legend=:bottomright)
    for i in 1:min(24, size(counts, 1))
        plot!(p_history, ts, counts[i, :]; seriestype=:steppost,
              color=:grey, alpha=0.14, linewidth=1, label=false)
    end

    trajectory_idx = rand(1:n_trajectories)
    plot!(p_history, ts, counts[trajectory_idx, :]; 
          seriestype=:steppost, color=cmagenta, linewidth=1.4, label=false)
    
    plot!(p_history, ts, vec(mean(counts; dims=1)); color=cturq, linewidth=3, label="Ensemble mean")
    vline!(p_history, [_t]; color=:gray, alpha=0.5, linestyle=:dash, label=false, linewidth=3)
    
    count_axis = 0:maximum(counts)
    probabilities = [mean(counts .== k) for k in count_axis]
    p_snapshot = bar(count_axis, probabilities; orientation=:horizontal,
                      color="#D5DCE2", linecolor=:white, label=false,
                      xlabel="Probability", title="t = $(_t) min", titlefontsize=12)

    if highlight_bar
        scatter!(p_snapshot, [-0.02], [snapshot_values[trajectory_idx]]; 
                 markershape=:hline, markersize=16, markerstrokewidth = 5, color=cmagenta, alpha=0.6, label=false)
    end
        
    hline!(p_snapshot, [mean(snapshot_values)]; color=cturq, alpha=0.5, linestyle=:dash, label=false, linewidth=2)
    
    xlim = highlight_bar ? -0.04 : -0.005
    xlims!(p_snapshot, (xlim, max(0.02, maximum(probabilities)*1.1)))
    ylims!(p_snapshot, (-1, maximum(counts)))
    
    plot!(p_snapshot; xticks=0.0:0.05:max(0.15, maximum(probabilities)*1.1))
    plot(p_history, p_snapshot; layout=grid(1, 2, widths = [0.6, 0.4]), link=:y, size=(1000, 400), bottom_margin=5mm, left_margin=5mm)
end

In [ ]:
plot_dynamics(ms, 3.0)

The gray dashed vertical line indicates the timepoint at which we record the snapshot distribution of mRNA counts across all SSA trajectories (individual cells), given by `n_trajectories`. The resulting histogram is plotted on the right, with the horizontal green dashed line representing the mean mRNA level at the chosen timepoint. The little horizontal bar to the left of the histogram indicates to which histogram bin does the highlighted (randomly chosen) trajectory belong to. 

### Exercises

* Try estimating the mRNA distribution at different simulation times. How does the distribution and its moments evolve in time?
* What effect does changing $\rho_m$ have on the trajectories of the system? What about $\delta_m$?
* Try varying `n_trajectories`. How do the time-dependent mean and the count distribution change with the number of simulations? How many simulations do we need for a reasonably small uncertainty in our estimates?
* By running $1000$ (or more) simulations, can you estimate the average number of mRNA, its variance and the Fano factor (variance divided by the mean) at the last timepoint (`tmax`)?
* Which statistical distribution does the simulated mRNA distribution correspond to?

## Transcriptional bursting and gene regulation

Our first model assumes that transcription is constitutive, that is, that the gene is always available and transcribing at the same rate. To better capture empirical gene expression measurements, which show alternating active and inactive periods of transcription, we now allow the promoter to switch between two states.

### Experiments

Transcriptional bursts — episodes of transcription separated by inactive periods — have been observed in living cells across different biological systems [[1](#1), [2](#2), [3](#3)].

This can observed on the level of individual fluorescence traces from live-cell imaging experiments (adapted from [[2](#2)]):

<img src="figures/burst_traces.png" alt="Bursty transcription traces" width="400"/>\
Similarly, the corresponding empirical distributions appear overdispersed and clearly diverge from the best-fit Poisson distribution (adapted from [[3](#3)]):

<img src="figures/bursty_example_histogram.png" alt="Bursty transcription example histogram" width="400"/>\
On the discrete RNA level, we expect a number of transcription events to occur within a short burst of activity, followed by a long quiescent period:

![Bursty behaviour](figures/burst_example.svg)\
<small id="1">1 Golding et al. Real-Time Kinetics of Gene Activity in Individual Bacteria. [Cell, 123, 1025-1036, (2005)](https://doi.org/10.1016/j.cell.2005.09.031). [^](#1)</small>\
<small id="2">2 Suter et al. Mammalian Genes Are Transcribed with Widely Different Bursting Kinetics. [Science 332,472-474 (2011)](https://doi.org/10.1126/science.1198817). [^](#2)</small>\
<small id="3">3 Raj et al. Nature, Nurture, or Chance: Stochastic Gene Expression and Its Consequences. [Cell, 135, 216-226 (2008)](10.1016/j.cell.2008.09.050). [^](#3)</small>

**Can a simple change to our constitutive model generate this behaviour?**

### Telegraph model

Most genes in real cells are regulated, and this can be represented in simplistic terms by assuming that a gene is either transcriptionally active (on) or inactive (off). (Note that these are *effective* promoter states, and we currently ignore transcription-factor binding, chromatin changes, polymerase recruitment etc...). We model this by separating $G$ into two species, $G_{on}$ and $G_{off}$, and now allow switching between the states with rates $\sigma_{on}$ and $\sigma_{off}$ — this model of gene expression is also known as the telegraph model.

<img src="figures/telegraph_model.png" alt="Telegraph model" width="500"/>

In [ ]:
rn = @reaction_network begin
    σ_on, G_off --> G_on      # gene activation
    σ_off, G_on --> G_off     # gene inactivation
    
    ρ_m, G_on --> G_on + M    # mRNA transcription
    δ_m, M --> 0              # mRNA degradation
end

We have to update our parameters and initial conditions. We start with a gene that rarely switches off once activated. Your next exercise will be to change the switching rates so that distinct bursts become visible.

In [ ]:
# Model parameters
p = [ :σ_on => 0.5, # TODO: make this poissonian like
      :σ_off => 0.001,
      :ρ_m => 5, 
      :δ_m => 1.0 ]

# Start with the gene in the off state
u0 = [ :G_off => 1, :G_on => 0, :M => 0 ];

As before we can convert this into a jump differential equation and solve it:

In [ ]:
jprob = JumpProblem(rn, u0, (0, tmax), p);

In [ ]:
sol = solve(jprob);

In [ ]:
plot(sol.t, sol[:M], 
     xlabel="Time", 
     ylabel="Molecules",
     label="M", 
     linetype=:steppost,
     linewidth=2)

plot!(sol.t, 
      sol[:G_on], 
      label="G_on", 
      linetype=:steppost,
      linewidth=2)

We can plot the mRNA and promoter state dynamics using the following (more elaborate) `plot_telegraph_trace` function:

In [ ]:
# NOTE: the helper plotting function is just for nicer visuals -- run this cell but feel free to ignore the internals

function plot_telegraph_trace(sol; title="")
    t, m, g = sol.t, sol[:M], sol[:G_on]
    birth_indices = findall(diff(m) .== 1) .+ 1
    birth_times = t[birth_indices]
    limits = (first(t), last(t))

    # Select the first complete visible bursty episode (with at least 3 mRNA produced).
    # Fall back to the first complete visible episode, which may produce zero RNA.
    episodes = Tuple{Float64,Float64,Int}[]
    start_time = NaN
    for i in 2:length(t)
        if g[i-1] == 0 && g[i] == 1
            start_time = t[i]
        elseif g[i-1] == 1 && g[i] == 0 && !isnan(start_time)
            if start_time >= limits[1] && t[i] <= limits[2]
                n = count(x -> start_time <= x <= t[i], birth_times)
                push!(episodes, (start_time, t[i], n))
            end
            start_time = NaN
        end
    end
    selected = findfirst(e -> e[3] >= 3, episodes)
    selected = isnothing(selected) && !isempty(episodes) ? 1 : selected

    p1 = plot(t, g; seriestype=:steppost, color=cmagenta,
              label=false, ylabel="Promoter", yticks=([0, 1], ["OFF", "ON"]),
              ylims=(-0.1, 1.55), xlims=limits, title=title, linewidth=2)
    p2 = scatter(birth_times, ones(length(birth_times));
                 markershape=:vline, markersize=20, markerstrokewidth = 2, color=cmagenta, alpha=0.6,
                 label=false, ylabel="RNA births", yticks=false,
                 ylims=(0.95, 1.05), xlims=limits, linewidth=1)
    p3 = plot(t, m; seriestype=:steppost, color=cmagenta,
              label=false, ylabel="RNA count", xlabel="Time (min)", xlims=limits, linewidth=2)
    
    if !isnothing(selected)
        a, b, n = episodes[selected]
        for panel in (p1, p2, p3)
            vspan!(panel, [a, b]; color=cturq, alpha=0.13, label=false)
        end
        label_x = limits[1] + 0.5*(limits[2]-limits[1])
        annotate!(p1, label_x, 1.35,
                  text("Burst duration: $(round(b-a; digits=2)) min", 9, cturq))
        annotate!(p2, label_x, 1.48,
                  text("$n RNA births during the burst", 9, cturq))
    end
    
    return plot(p1, p2, p3; layout=(3, 1), link=:x, size=(900, 680))
end

In [ ]:
plot_telegraph_trace(sol)

### Exercises:

* Can you find a parameter regime that reproduces transcriptional bursting?\
You might find the following code snippet useful for plotting:
    ```julia
    p = [:σ_on => 0.5,    # change the parameters
         :σ_off => 0.001,
         :ρ_m => 5, 
         :δ_m => 1.0 ]
    tspan = (0.0, 10.0)   # change the simulation timespan
    
    jprob = remake(jprob; p, tspan) # remake the jump problem with updated p and tspan
    plot_telegraph_trace(solve(jprob)) # solve the jump problem and plot the simulation trace
    ```

* Can you estimate the probability that the gene is on at time $t = 10$? How does changing the switching rates $\sigma_{on}$ and $\sigma_{off}$ affect this? Note that you can reuse code from earlier (in particular `plot_dynamics`).
* Consider the variance and the Fano factor of the mRNA count distribution. How does it change when the system is bursty (compared to the constitutive model)?

### Same mean, different dynamics

Here, we compare how the mRNA dynamics in the telegraph model change as we tune the promoter switching timescale. Namely, we compare *slow* and *fast* switching regimes.

In [ ]:
n_trajectories = 10000
ts = 0:1.0:150.0

counts_slow = Matrix{Int}(undef, n_trajectories, length(ts))
counts_fast = Matrix{Int}(undef, n_trajectories, length(ts))

p_slow = [:σ_on => 0.05,
          :σ_off => 1.0,
          :ρ_m => 10.0, 
          :δ_m => 0.1 ]

p_fast = [:σ_on => 20 * 0.05, # note the rescaled (faster) switching
          :σ_off => 20 * 1.0,
          :ρ_m => 10.0, 
          :δ_m => 0.1 ];

In [ ]:
jprob = remake(jprob, p=p_slow, tspan=(ts[1], ts[end]))
plot_telegraph_trace(solve(jprob))

In [ ]:
jprob = remake(jprob, p=p_fast, tspan=(ts[1], ts[end]))
plot_telegraph_trace(solve(jprob))

In [ ]:
jprob = remake(jprob, p=p_slow)
for i in 1:n_trajectories
    sol_SSA = solve(jprob)
    m = sol_SSA(ts, idxs=:M) # record the mRNA count at time t = tmax
    counts_slow[i, :] = m
end
        
jprob = remake(jprob, p=p_fast)
for i in 1:n_trajectories
    sol_SSA = solve(jprob)
    m = sol_SSA(ts, idxs=:M) # record the mRNA count at time t = tmax
    counts_fast[i, :] = m
end

_idx = length(ts)
plot_slow = plot_dynamics(counts_slow, ts[_idx], title="slow switching")
ylims!(plot_slow, 0, 50)
plot_fast = plot_dynamics(counts_fast, ts[_idx], title="fast switching")

println("Mean at t = $(ts[_idx]) (slow switching) is $(mean(counts_slow[:, end]))")
println("Mean at t = $(ts[_idx]) (fast switching) is $(mean(counts_fast[:, end]))")
plot(plot_slow, plot_fast, layout=(2, 1), size=(800, 700)) 

In [ ]:
plot_slow = histogram(counts_slow[:, end], 
                       bins=0:maximum(counts_slow),
                       normalize=:probability,
                       xlabel="Molecules", 
                       ylabel="Probability", 
                       label="mRNA",
                       title="slow switching",
                       color=cturq, alpha=0.8)
vline!(plot_slow, [mean(counts_slow[:, end])], color=cmagenta, alpha=0.8, linewidth=2, label="mean")

plot_fast = histogram(counts_fast[:, end], 
                       bins=0:maximum(counts_fast),
                       normalize=:probability,
                       xlabel="Molecules", 
                       ylabel="Probability", 
                       label=nothing,
                       title="fast switching",
                       color=cturq, alpha=0.8) 
vline!(plot_fast, [mean(counts_fast[:, end])], color=cmagenta, alpha=0.8, linewidth=2, label="mean")

plot(plot_slow, plot_fast, size=(700, 300), left_margin=5mm, bottom_margin=5mm)

We increased both switching rates by the same factor, preserving the long-run fraction of time the gene is active and therefore its long-run mean expression. However, the timing of transcription and the variability between cells change. Similar average expression can therefore arise from different regulatory dynamics. As a result, looking at the mean expression is often not enough; noise can help us distinguish the underlying kinetics.

## Splicing dynamics

So far, all RNA has been represented by a single species. One simple extension in this direction is to explicitly model the splicing dynamics of RNA.

<div align="center">
  <img src="figures/spliced_unspliced_scRNAseq.png" alt="dropled-based scRNAseq" width="800">
  <br>
  <span><small> Adapted from [1]. </small></span>
</div>

This is also important for single-cell data analysis, as droplet-based single-cell RNA sequencing protocols provide measurements both for unspliced and spliced (nascent and mature) RNA counts. The presence of spliced-unspliced counts can be leveraged to estimate the RNA velocity of cells, i.e. the time derivative of the gene expression state, and predict the future state of cells [[1](#1)]. RNA velocity-based computational frameworks often use a transcriptional model that incorporates splicing.

<div align="center">
  <img src="figures/RNA_velocity.png" alt="RNA velocity" width="800">
  <br>
  <span><small> Adapted from [1]. </small></span>
</div>

<small id="1">1 La Manno et al. RNA velocity of single cells. [Nature 560, 494–498 (2018)](https://www.nature.com/articles/s41586-018-0414-6). [^](#1)</small>

Most of such methods use a simpler version of the model we introduce next: they describe transcription,
the conversion of unspliced into spliced RNA, and RNA degradation, without explicitly
modelling stochastic promoter switching. We will now extend our telegraph model to
include these same RNA-processing steps, allowing us to explore how transcriptional bursts
affect the dynamics of both unspliced and spliced RNA.

<div align="center">
  <img src="figures/telegraph_model_w_splicing.png" alt="Telegraph model with splicing" width="700">
</div>

**Exercise:** Modify the telegraph model to include the unspliced RNA species $U$ and the splicing reaction with rate parameter $\gamma_m$.

In [ ]:
rn = @reaction_network begin
    σ_on, G_off --> G_on      # gene activation
    σ_off, G_on --> G_off     # gene inactivation
    
    ρ_m, G_on --> G_on + U    # produce unspliced RNA
    γ_m, U --> M              # produce spliced RNA
    
    δ_m, M --> 0              # mRNA degradation
end

In [ ]:
# Model parameters
p = [ :σ_on => 0.5, 
      :σ_off => 50.0,
      :ρ_m => 500.0,
      :γ_m => 2.0,
      :δ_m => 1.0 ]

# Start with the gene in the off state
u0 = [ :G_off => 1, :G_on => 0, :U => 0, :M => 0 ];

In [ ]:
jprob = JumpProblem(rn, u0, (0, 50.0), p);

In [ ]:
sol = solve(jprob);

In [ ]:
plot(sol.t, 
     sol[:U], 
     xlabel="Time", 
     ylabel="Molecules",
     label="unspliced RNA", 
     linetype=:steppost,
     linewidth=2)

plot!(sol.t, 
      sol[:M], 
      label="spliced RNA", 
      linetype=:steppost,
      linewidth=2)

Next, run the following cell to recover the classic almond shape in the spliced-unspliced RNA space!

In [ ]:
u0 = [:G_off => 1, :G_on => 0, :U => 0, :M => 0]
tspan = (0.0, 400.0)
times = collect(range(tspan...; step=0.2))

p = (σ_on = 0.025,
     σ_off = 0.025,
     ρ_m = 30.0,
     γ_m = 0.2,
     δ_m = 0.1)

prob = JumpProblem(
    rn, u0, tspan,
    # Slow switching and higher RNA counts help reveal the phase-plane structure.
    [:σ_on => p.σ_on,
     :σ_off => p.σ_off,
     :ρ_m => p.ρ_m,
     :γ_m => p.γ_m,
     :δ_m => p.δ_m];
    save_positions=(false, false)
)

n_cells = 500
n_show = min(3, n_cells)

U = Matrix{Float64}(undef, length(times), n_cells)
M = similar(U)

for i in 1:n_cells
    sol = solve(prob; saveat=times)
    U[:, i] .= sol[:U]
    M[:, i] .= sol[:M]
end

mean_U = vec(mean(U; dims=2))
mean_M = vec(mean(M; dims=2))

# RNA time series
a = plot(
    xlabel="Time",
    ylabel="RNA",
    #title="Individual cells and population mean",
    titlefontsize=12,
    legend=:topright
)

# Unspliced–spliced phase portrait.
b = plot(
    xlabel="Spliced RNA",
    ylabel="Unspliced RNA",
    #title="Trajectories and final-time cell snapshot",
    titlefontsize=12,
    legend=:topleft
)

# Highlight a few individual trajectories
for i in 1:n_show
    plot!(a, times, U[:, i];
          color=cturq, alpha=0.15, linewidth=1, seriestype=:steppost, label=false)
    plot!(a, times, M[:, i];
          color=cmagenta, alpha=0.15, linewidth=1, seriestype=:steppost, label=false)
    plot!(b, M[:, i], U[:, i];
          color=cmagenta, alpha=0.15, linewidth=1, label=false)
end

# Final-time snapshot over the cell population
scatter!(b, M[end, :], U[end, :];
         color=cmagenta,
         markersize=4,
         markerstrokewidth=0,
         alpha=0.4,
         label="Cells at t = $(round(times[end]; digits=1)) min")

# Population mean over time
plot!(a, times, mean_U;
      color=cturq, linewidth=3, label="Mean unspliced")

plot!(a, times, mean_M;
      color=cmagenta, linewidth=3, label="Mean spliced")

# Zero drift line: γ_m U − δ_m M = 0.
xmax = 1.1 * max(1.0, maximum(M[:, 1:n_show]), maximum(M[end, :]), maximum(mean_M) )
plot!(b, [0.0, xmax], [0.0, (p.δ_m / p.γ_m) * xmax];
      color=:gray, linestyle=:dash, alpha=0.5, label="Zero spliced-RNA drift")

# Mean phase trajectory and its final point.
plot!(b, mean_M, mean_U;
      color=:gray, linewidth=3, alpha=0.7, label="Population mean")

scatter!(b, [mean_M[end]], [mean_U[end]];
         color=:gray, alpha=0.95, markershape=:diamond, markersize=7, markerstrokewidth=0, label="Final population mean")

plot(a, b; layout=(1, 2), size=(1050, 350), bottom_margin=5mm, left_margin=5mm)

Note that our stochastic simulations lead to substantial deviations from the expected deterministic shape. Nevertheless, the distribution of the scatter points (essentially a time snapshot for a cell population) can be used succesfully to fit RNA velocity-based models.

### Exercises

* How does changing the promoter switching rates affect the system behaviour?
* What happens if you speed up splicing?

## Adding protein

So far we have followed RNA. We now add the protein it produces, allowing us to explore how transcriptional fluctuations propagate to a downstream production.

<div align="center">
  <img src="figures/telegraph_model_w_translation.png" alt="Telegraph model with protein translation" width="500">
</div>

This section is an **exercise**. The goal is to add protein to the previous reaction system (note that here we ignore the unspliced RNA again for simplicity). We assume that the protein (which we denote $P$) is translated from mRNA at rate $\rho_p$, and degraded with rate $\delta_m$. Add these two reactions to the previous reaction network of the telegraph model, simulate the system and plot the solution. You can use the parameter values $\rho_p = 10$ and $\delta_p = 0.1$, or come up with your own. Play around with parameter values and initial conditions to see how the system behaves!

## Feedback

<div align="center">
  <img src="figures/feedback_model.png" alt="Positive feedback loop" width="500">
</div>

Our previous model of gene expression assumes that promoter switching rates are fixed, so that the gene switches on and off at random times without us specifying exactly how that happens. What changes if the protein produced by the gene affects those rates? In reality, this is a very deliberate process. Many genes self-regulate, that is, they promote or repress their own transcription. We can modify our previous system to model a positive feedback loop, by assuming that our protein $P$ binds to its own transcription start site and prevents deactivation of the gene. We implement this by incorporating the protein concentration in the gene inactivation mechanism, using a repressive Michaelis-Menten equation (the function `mmr` in Catalyst):

In [ ]:
rn = @reaction_network begin
    σ_on, G_off --> G_on      # gene activation
    
    # gene inactivation now depends on protein numbers
    mmr(P, σ_off, K), G_on --> G_off
    
    ρ_m, G_on --> G_on + M    # transcription
    δ_m, M --> 0              # mRNA degradation

    ρ_p, M --> M + P          # translation
    δ_p, P --> 0              # protein degradation
end

Here the gene inactivation rate is

$$ \sigma_{off} \cdot \frac{K}{K + P} $$

which is inhibiting for high $P$. For $P = 0$, this is simply $\sigma_{off}$, so nothing changes. As $P$ increases, the rate decreases; at concentration $K$, inactivation is only 50% as fast, and as $P$ increases beyond that, the reaction rate approaches $0$. 

In [ ]:
# Model parameters
p = [ :σ_on => 1,
      :σ_off => 1,
      :K => 40,
    
      :ρ_m => 5, 
      :δ_m => 0.5,

      :ρ_p => 5,
      :δ_p => 1 ]

# Start with the gene in the off state
u0 = [ :G_off => 1, :G_on => 0, :M => 0, :P => 0 ];

tmax = 50.;

As before we can convert this into a jump differential equation and solve it:

In [ ]:
jprob = JumpProblem(rn, u0, (0, tmax), p);

In [ ]:
sol = solve(jprob);

In [ ]:
plot(sol.t, sol[:M], 
     xlabel="Time", 
     ylabel="mRNA counts",
     label="M", 
     linetype=:steppost)

plot!(sol.t, sol[:G_on], 
      label="G_on", 
      linetype=:steppost)

In [ ]:
plot(sol.t, sol[:P], 
     xlabel="Time", 
     ylabel="Protein counts",
     label="P", 
     linetype=:steppost)

In [ ]:
# Number of simulations to perform
n_trajectories = 100
# Time points for saving the system state
ts = 0:0.1:tmax
# Matrix to store the count time series from each simulation
counts = Matrix{Int}(undef, n_trajectories, length(ts))

for i in 1:n_trajectories
    sol_SSA = solve(jprob)
    m = sol_SSA(ts, idxs=:P) # record the mRNA count at time t = tmax
    counts[i, :] = m
end

In [ ]:
plot_dynamics(counts, 10.0; ylabel="Protein counts", highlight_bar=false)

### Exercises
* Note the peak at zero in the protein distribution. What could be causing the appearance of bimodality?
  
* What are other ways in which positive autoregulation could be built into this system? Think about examples of positive feedback you have encountered in biological systems.

* How could you incorporate negative feedback into this system? How does the system behave in this case?


## Adding cell cycle dynamics

<div align="center">
  <img src="figures/cell_cycle_model.png" alt="Cell cycle model" width="300">
</div>

Cells growth and division can shape gene expression in many ways. For example, as a cell grows and its volume increases, it must compensate for differences due to changing molecular concentrations, and, similarly, following DNA replication, it must alter the transcriptional activity to account for duplicated gene copy numbers [[1](#1)]. In addition, division timing and molecular partitioning between daughter cells can further shape gene-product distributions.

<small id="1">1 Padovan-Merhar et al. Single mammalian cells compensate for differences in cellular volume and DNA copy number through independent global transcriptional mechanisms. [Molecular cell 58.2 (2015): 339-352](10.1016/j.molcel.2015.03.005).</small>

Here, we will implement a rudimentary form of the cell cycle for our gene expression model by implementing DNA replication and molecule partitioning upon division. This can be implemented in Catalyst by adding callbacks at times when the gene copy number doubles and when our cell divides - by choosing one daughter cell at random at each division, we will simulate a lineage. For this, we have to specify the cell-cycle length $T$ and what happens during division. In our case, we will divide mRNA and protein numbers by half - more precisely, we assume that each molecule has a 50% chance of ending up in the each daughter cell.


In [ ]:
rn = @reaction_network begin
    σ_on, G_off --> G_on      # gene activation
    σ_off, G_on --> G_off     # gene inactivation
    
    ρ_m, G_on --> G_on + M    # transcription
    δ_m, M --> 0              # mRNA degradation

    ρ_p, M --> M + P          # translation
    δ_p, P --> 0              # protein degradation
end

In [ ]:
# Model parameters
p = [ :σ_on => 0.3,
      :σ_off => 0.1,
      :ρ_m => 4.0, 
      :δ_m => 1.0,
      :ρ_p => 4.0, 
      :δ_p => 1.0]

# Start with the gene in the off state
u0 = [ :G_off => 1, :G_on => 0, :M => 0, :P => 0 ];
tmax = 55.0

jprob = JumpProblem(rn, u0, (0, tmax), p);

In [ ]:
using Distributions

# The argument `current_sol` contains information about the current cell state
function replicate!(current_sol)
    current_sol[:G_on] = 2 * current_sol[:G_on]
    current_sol[:G_off] = 2 * current_sol[:G_off]
end;

function divide!(current_sol)
    # mRNA and protein are divided randomly between the daughters
    current_sol[:M] = rand(Binomial(current_sol[:M], 0.5))
    current_sol[:P] = rand(Binomial(current_sol[:P], 0.5))

    # Pick one of the two genes to inherit
    if 2 * rand() <= current_sol[:G_on]
        current_sol[:G_on] = 1
        current_sol[:G_off] = 0
    else
        current_sol[:G_on] = 0
        current_sol[:G_off] = 1
    end
end;

T_cellcycle = 20.;

[Callbacks](https://docs.sciml.ai/DiffEqDocs/stable/features/callback_functions/) are very simple to implement in DifferentialEquations. Using the built-in callback library, a simple addition to our `solve` call will do the trick:

In [ ]:
using DiffEqCallbacks

cb_rep = PeriodicCallback(replicate!, T_cellcycle; phase = T_cellcycle / 2)
cb_div = PeriodicCallback(divide!, T_cellcycle)

sol = solve(jprob; callback=CallbackSet(cb_rep, cb_div));

In the plots below we have marked the division times explicitly. While they are not easy to read off from the mRNA plots due to their volatility, they are more noticeable when looking at allolactose concentrations.

In [ ]:
plot(sol.t, sol[:M], 
           xlabel="Time", 
           ylabel="Molecules",
           label="M", 
           linewidth=1,
           linetype=:steppost)

plot!(sol.t, 
      sol[:P], 
      label="P", 
      linetype=:steppost)

plot!(sol.t, 
      sol[:G_on], 
      label="G_on", 
      linetype=:steppost)

vline!([ T_cellcycle, 2T_cellcycle ], color=cmagenta, alpha=0.8, linestyle=:dash, linewidth=1.5, label="Cell division")

## Outlook

We have moved from assumptions about biochemical reactions to predictions for trajectories and distributions across cells. These predictions help us ask which mechanisms could explain observed patterns and how the system might respond to a perturbation. The appropriate level of detail depends on the biological question and the measurements available.

Beyond this, there are countless ways to build more realistic gene expression models, including some of the aspects below:

### Gene regulatory networks

So far, we have only focussed on modelling the expression of a single gene. However, genes do not operate in isolation: their RNA and protein products can regulate the expression of other genes, forming highly interconnected gene regulatory networks. Understanding how cellular behaviour emerges from these interactions requires us to extend the models to incorporate multiple genes and the regulatory links between them.

### Chromatin modelling

The telegraph models each gene as being active or inactive, and switching between these two states randomly. While epigenetic factors like chromatin modelling and DNA methylation can determine what genes are transcribed, real promoter activation dynamics are much more complicated than suggested by the telegraph model. Developing more realistic models could explicitly model chromatin state at relevant locations of the genome, inferred from data such as ATAC-seq.

### Realistic cell cycle modelling

Cells undergo different phases (G1, S, G2/M etc.) during their lifetime, which we did not need to model explicitly in our simple model. Cell cycle progression and especially division do not occur at fixed times, but are controlled by complex molecular machinery that is still to be explored. Furthermore, division may be asymmetric (as in budding yeast), and partitioning of molecules may not be entirely binomial. A more realistic depiction could include these aspects instead of requiring division by half every 20 minutes.

### Cell size

Many intracellular processes including gene expression and metabolism are strongly affected by cell size, which we did not mention here. We could model volume $V$ by coupling the above reaction network to an ordinary differential equation of the form

$$ \frac{d}{dt} V = \gamma V $$

where $\gamma$ is the growth rate of a cell. Of course, during division the daughter cell inherits a fraction of the mother's volume. We could even make volume growth itself stochastic (turning this into a stochastic differential equation) and have cells divide once they reach a fixed volume - this approximates the cell cycle dynamics of fission yeast.

### Spatial modelling

Cells are not just little blobs in which stuff happens - many of them are very strictly organised inside. This is especially the case for Eukaryotes, which generally have a nucleus, a cytosol and countless organelles like mitochondria. A more realistic model of eukaryotic gene expression could distinguish e.g. the nucleus (in which mRNA transcription and folding happens) and the cytosol (where most proteins are translated). This is also important for complex signal pathways in which information has to be relayed from the cell surface to the nucleus, and vice versa.

### More cells

Cells like to communicate with each other. To model tissues and organisms, it is often necessary to combine many cells into one network of interactions, allowing them to exchange information and respond collectively to the environment. Catalyst now provides [functionality for spatial modelling](https://docs.sciml.ai/Catalyst/stable/spatial_modelling/discrete_space_reaction_systems/) that can be used to construct such models with relative ease.